In [ ]:
import tensorflow as tf
import pandas as pd
import numpy as np
import seaborn as sns
import sqlite3
import matplotlib as mpl
import matplotlib.pyplot as plt
import keras_tuner as kt

import copy
import random
import math

from sqlalchemy import create_engine
from tensorflow.keras import layers
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import CSVLogger
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans

from datetime import datetime

In [ ]:
dbConnectionString = "sqlite:///baseball_info.db"
engine = create_engine(dbConnectionString)

In [ ]:
timeSeriesHittingQuery = "SELECT SeasonStatsHitting.playerId,season,Bios.dob,plateAppearances,atBats,runs,hits,singles,doubles,triples,homeRuns,rbis,sacHits,sacFlies,hitByPitch,walks,intentionalWalks,strikeOuts,stolenBases,caughtStealing,groundedIntoDoublePlays,catcherInterference,battingAverage,onBasePercentage,sluggingPercentage,onBasePlusSlugging,runsPerTob,rbisPerBip,babip,spd,groundBallPercentage,flyBallPercentage,lineDrivePercentage,popUpPercentage,hardHitPercentage,barrelPercentage FROM SeasonStatsHitting INNER JOIN Bios on Bios.playerId = SeasonStatsHitting.playerId WHERE season BETWEEN 2015 and 2025 and season != 2020 ORDER BY SeasonStatsHitting.playerId,season"

df = pd.read_sql(timeSeriesHittingQuery, engine)

In [ ]:
def calculate_age_from_timestamps(birth_timestamp, current_timestamp):
    birth_date   = datetime.fromtimestamp(birth_timestamp)
    current_date = datetime.fromtimestamp(current_timestamp)

    age = current_date.year - birth_date.year

    # Adjust age if the birthday hasn't occurred yet in the current year
    if (current_date.month, current_date.day) < (birth_date.month, birth_date.day):
        age -= 1
    
    return age

In [ ]:
twentyFifteenDateString     = "2015-04-05 00:00:00"
twentySixteenDateString     = "2016-04-03 00:00:00"
twentySeventeenDateString   = "2017-04-02 00:00:00"
twentyEighteenDateString    = "2018-03-29 00:00:00"
twentyNineteenDateString    = "2019-03-20 00:00:00"
twentyTwentyOneDateString   = "2021-04-01 00:00:00"
twentyTwentyTwoDateString   = "2022-04-07 00:00:00"
twentyTwentyThreeDateString = "2023-03-30 00:00:00"
twentyTwentyFourDateString  = "2024-03-20 00:00:00"
twentyTwentyFiveDateString  = "2025-03-18 00:00:00"

# Parse the string into a datetime object
twentyFifteenDateString     = datetime.strptime(twentyFifteenDateString     , "%Y-%m-%d %H:%M:%S")
twentySixteenDateString     = datetime.strptime(twentySixteenDateString     , "%Y-%m-%d %H:%M:%S")
twentySeventeenDateString   = datetime.strptime(twentySeventeenDateString   , "%Y-%m-%d %H:%M:%S")
twentyEighteenDateString    = datetime.strptime(twentyEighteenDateString    , "%Y-%m-%d %H:%M:%S")
twentyNineteenDateString    = datetime.strptime(twentyNineteenDateString    , "%Y-%m-%d %H:%M:%S")
twentyTwentyOneDateString   = datetime.strptime(twentyTwentyOneDateString   , "%Y-%m-%d %H:%M:%S")
twentyTwentyTwoDateString   = datetime.strptime(twentyTwentyTwoDateString   , "%Y-%m-%d %H:%M:%S")
twentyTwentyThreeDateString = datetime.strptime(twentyTwentyThreeDateString , "%Y-%m-%d %H:%M:%S")
twentyTwentyFourDateString  = datetime.strptime(twentyTwentyFourDateString  , "%Y-%m-%d %H:%M:%S")
twentyTwentyFiveDateString  = datetime.strptime(twentyTwentyFiveDateString  , "%Y-%m-%d %H:%M:%S")

# Convert to Unix timestamp
twentyFifteenTimeStamp     = int(twentyFifteenDateString     .timestamp())
twentySixteenTimeStamp     = int(twentySixteenDateString     .timestamp())
twentySeventeenTimeStamp   = int(twentySeventeenDateString   .timestamp())
twentyEighteenTimeStamp    = int(twentyEighteenDateString    .timestamp())
twentyNineteenTimeStamp    = int(twentyNineteenDateString    .timestamp())
twentyTwentyOneTimeStamp   = int(twentyTwentyOneDateString   .timestamp())
twentyTwentyTwoTimeStamp   = int(twentyTwentyTwoDateString   .timestamp())
twentyTwentyThreeTimeStamp = int(twentyTwentyThreeDateString .timestamp())
twentyTwentyFourTimeStamp  = int(twentyTwentyFourDateString  .timestamp())
twentyTwentyFiveTimeStamp  = int(twentyTwentyFiveDateString  .timestamp())

seasonToStartTimeStamp = {}

seasonToStartTimeStamp[2015] = twentyFifteenTimeStamp   
seasonToStartTimeStamp[2016] = twentySixteenTimeStamp   
seasonToStartTimeStamp[2017] = twentySeventeenTimeStamp   
seasonToStartTimeStamp[2018] = twentyEighteenTimeStamp   
seasonToStartTimeStamp[2019] = twentyNineteenTimeStamp   
seasonToStartTimeStamp[2021] = twentyTwentyOneTimeStamp  
seasonToStartTimeStamp[2022] = twentyTwentyTwoTimeStamp  
seasonToStartTimeStamp[2023] = twentyTwentyThreeTimeStamp
seasonToStartTimeStamp[2024] = twentyTwentyFourTimeStamp 
seasonToStartTimeStamp[2025] = twentyTwentyFiveTimeStamp

In [ ]:
#drop players that cannot create a valid window (ie have not played multiple seasons)
hasMultipleSeasons = df["playerId"].value_counts() > 1
df = df[df["playerId"].isin(hasMultipleSeasons[hasMultipleSeasons].index)]

#drop players with no plater appearances
df.drop(df[df['plateAppearances'] == 0].index, inplace=True)

# columnsToCheck = ['runs', 'homeRuns', 'rbis', 'stolenBases', 'onBasePercentage']

# zeroRowsMask = (df[columnsToCheck] != 0).any(axis=1)

# df = df[zeroRowsMask]

#get player ages
df['dob'] = df.apply(lambda row: calculate_age_from_timestamps(row['dob'], seasonToStartTimeStamp[row['season']]), axis=1)

#logColumns     = df.columns[3:]
#df[logColumns] = np.log(df[logColumns] + 1)
#
# numericDf     = df.select_dtypes(include=np.number)
# categoricalDf = df.select_dtypes(exclude=np.number)

# mean = numericDf.mean()
# std  = numericDf.std()

# dfNormalized = (numericDf - mean) / std

# dfNormalized = pd.concat([categoricalDf, dfNormalized], axis=1)

In [ ]:
len(df)

In [ ]:
def getWindowedFeaturesAndLabels(frame, maxWindowSize):
    frameArray = frame.values.tolist()

    inputs = []
    labels = []
    
    r = 0

    curWindow = []

    playerId = frameArray[0][0]
    
    while r < len(frameArray):
        curPlayerId = frameArray[r][0]

        if curPlayerId != playerId:
            window = []
            
            windowR = 0

            while windowR <= len(curWindow):
                if len(window) == maxWindowSize or windowR == len(curWindow):
                    label = window.pop()

                    labelTest = copy.deepcopy(label)

                    relevantLabels = labelTest[5:6] + labelTest[10:12] + labelTest[18:19] + labelTest[23:24]

                    if len(window) < maxWindowSize - 1:
                        diff = maxWindowSize - 1 - len(window)

                        while diff > 0:
                            padding = [-10000.0] * 36
                            window.append(padding)

                            diff -= 1

                    windowArray = [inner_list[2:] for inner_list in window]

                    inputs.append(copy.deepcopy(windowArray))
                    labels.append(copy.deepcopy(relevantLabels))

                    window.append(label)
                    window.pop(0)

                if windowR < len(curWindow):
                    window.append(curWindow[windowR])
                
                windowR += 1

            curWindow = []
            playerId  = curPlayerId

        curWindow.append(frameArray[r])

        r += 1

    return inputs,labels

In [ ]:
fullFeatures,fullLabels = getWindowedFeaturesAndLabels(df, 4)

In [ ]:
def getNaiveLoss(frame, maxWindowSize):
    meanSquaredError = 0.0
    labelCount       = 0.0
    
    frameArray = frame.values.tolist()

    r = 0

    curWindow = []

    playerId = frameArray[0][0]
    
    while r < len(frameArray):
        curPlayerId = frameArray[r][0]

        if curPlayerId != playerId:
            window = []
            
            windowR = 0

            while windowR <= len(curWindow):
                if len(window) == maxWindowSize or windowR == len(curWindow):
                    label = window.pop()

                    labelTest = copy.deepcopy(label)

                    relevantLabels = labelTest[5:6] + labelTest[10:12] + labelTest[18:19] + labelTest[23:24]

                    labelRuns        = relevantLabels[0]
                    labelHomeRuns    = relevantLabels[1]
                    labelRbis        = relevantLabels[2]
                    labelStolenBases = relevantLabels[3]
                    labelsObp        = relevantLabels[4]

                    if len(window) < maxWindowSize - 1:
                        diff = maxWindowSize - 1 - len(window)

                        while diff > 0:
                            padding = [-10000.0] * 36
                            window.append(padding)

                            diff -= 1

                    windowArray = [inner_list[2:] for inner_list in window]

                    lastWindow = windowArray[0]

                    relevantWindowLabels = lastWindow[5:6] + lastWindow[10:12] + lastWindow[18:19] + lastWindow[25:26]

                    naiveRuns        = relevantWindowLabels[0]
                    naiveHomeRuns    = relevantWindowLabels[1]
                    naiveRbis        = relevantWindowLabels[2]
                    naiveStolenBases = relevantWindowLabels[3]
                    naivesObp        = relevantWindowLabels[4]

                    meanSquaredError += pow(naiveRuns        - labelRuns       , 2)
                    meanSquaredError += pow(naiveHomeRuns    - labelHomeRuns   , 2)
                    meanSquaredError += pow(naiveRbis        - labelRbis       , 2)
                    meanSquaredError += pow(naiveStolenBases - labelStolenBases, 2)
                    meanSquaredError += pow(naivesObp        - labelsObp       , 2)

                    labelCount += 1.0
                    
                    window.append(label)
                    window.pop(0)

                if windowR < len(curWindow):
                    window.append(curWindow[windowR])
                
                windowR += 1

            curWindow = []
            playerId  = curPlayerId

        curWindow.append(frameArray[r])

        r += 1

    meanSquaredError = meanSquaredError / labelCount

    return meanSquaredError

In [ ]:
naiveMse = getNaiveLoss(df, 4)

naiveMse

In [ ]:
fullFeaturesArray = np.array(fullFeatures)
fullLabelsArray   = np.array(fullLabels)

In [ ]:
fullFeaturesArray.shape

In [ ]:
runBins              = np.linspace(min(fullLabelsArray[:,0]), max(fullLabelsArray[:,0]), num=4)
homeRunsBins         = np.linspace(min(fullLabelsArray[:,1]), max(fullLabelsArray[:,1]), num=4)
rbiBins              = np.linspace(min(fullLabelsArray[:,2]), max(fullLabelsArray[:,2]), num=4)
stolenBaseBins       = np.linspace(min(fullLabelsArray[:,3]), max(fullLabelsArray[:,3]), num=4)
onBasePercentageBins = np.linspace(min(fullLabelsArray[:,4]), max(fullLabelsArray[:,4]), num=4)

In [ ]:
runsBinned             = np.digitize(fullLabelsArray[:,0], runBins, right=True)
homeRunsBinned         = np.digitize(fullLabelsArray[:,1], homeRunsBins, right=True)
rbisBinned             = np.digitize(fullLabelsArray[:,2], rbiBins, right=True)
stolenBasesBinned      = np.digitize(fullLabelsArray[:,3], stolenBaseBins, right=True)
onBasePercentageBinned = np.digitize(fullLabelsArray[:,4], onBasePercentageBins, right=True)

In [ ]:
fullLabelsArrayBinned = np.column_stack((runsBinned, homeRunsBinned, rbisBinned, stolenBasesBinned, onBasePercentageBinned))

In [ ]:
binCount = {}

for label in fullLabelsArrayBinned:
    labelString = str(label[0]) + str(label[1]) + str(label[2]) + str(label[3]) + str(label[4])

    if labelString in binCount:
        binCount[labelString] = binCount[labelString] + 1
    else:
        binCount[labelString] = 1

In [ ]:
sortedBinCount = dict(sorted(binCount.items(), key=lambda item: item[1], reverse=True))

len(sortedBinCount)

In [ ]:
labels = list(sortedBinCount.keys())
count  = list(sortedBinCount.values())

plt.figure(figsize=(16, 12))
bar = plt.bar(labels, count)

_ = plt.xticks(labels, rotation="vertical")
_ = plt.bar_label(bar, count)

In [ ]:
binIndexes = []

for label in labels:
    labelNumber = int(label)
    
    labelArray = []
    
    while labelNumber > 0:
        digit = labelNumber % 10
        
        labelArray.insert(0, digit)
        
        labelNumber = int(labelNumber / 10)
        
    while len(labelArray) < 5:
        labelArray.insert(0, 0)
        
    binIndexes.append(labelArray)

In [ ]:
def addNoiseToPoints(point, label):
    noisyPoint = np.copy(point)
    
    mean               = 0
    stdDevWholeNumbers = 1
    stdDevPercentages  = 0.01
    
    wholeNumbersMask = np.zeros_like(noisyPoint, dtype=bool)
    
    allZerosForSecondWindow = (noisyPoint[34:66] == -10000.0).all()
    allZerosForThirdWindow  = (noisyPoint[66:  ] == -10000.0).all()
    
    wholeNumbersMask[:20  ] = True
    
    #print(allZerosForSecondWindow)
    #print(allZerosForThirdWindow)
    
    if allZerosForSecondWindow != True:
        wholeNumbersMask[34:54] = True
        
    if allZerosForThirdWindow != True:
        wholeNumbersMask[68:88] = True
    
    percentageNumbersMask = np.zeros_like(noisyPoint, dtype=bool)
    
    percentageNumbersMask[20:34] = True
    
    if allZerosForSecondWindow != True:
        percentageNumbersMask[54:68] = True
        
    if allZerosForThirdWindow != True:
        percentageNumbersMask[88:  ] = True
    
    wholeNumbersToChange = np.sum(wholeNumbersMask)
    percentagesToChange  = np.sum(percentageNumbersMask)
    
    #print(wholeNumbersToChange.shape)
    
    wholeNumberNoise = np.random.normal(mean, stdDevWholeNumbers, size=wholeNumbersToChange)
    percentageNumberNoise = np.random.normal(mean, stdDevPercentages, size=percentagesToChange)
    
    #print(percentageNumberNoise)
    
    noisyPoint[wholeNumbersMask     ] = np.maximum(0, noisyPoint[wholeNumbersMask     ] + wholeNumberNoise)
    noisyPoint[percentageNumbersMask] = np.maximum(0, noisyPoint[percentageNumbersMask] + percentageNumberNoise)
    
    for i in range(len(noisyPoint)):
        if wholeNumbersMask[i] == True:
            noisyPoint[i] = math.ceil(noisyPoint[i])
            
    startAge = noisyPoint[0]
    
    noisyPoint[34] = startAge + 1
    noisyPoint[68] = startAge + 2
            
    noisyLabel = np.copy(label)
    
    #wholeNumbersLabelMask      = np.zeros_like(noisyLabel, dtype=bool)
    #percentageNumbersLabelMask = np.zeros_like(noisyLabel, dtype=bool)
    #
    #wholeNumbersLabelMask     [:4] = True
    #percentageNumbersLabelMask[4:] = True
    #
    #wholeNumbersLabelToChange      = np.sum(wholeNumbersLabelMask)
    #percentageNumbersLabelToChange = np.sum(percentageNumbersLabelMask)
    #
    #wholeNumbersLabelNoise      = np.random.normal(mean, stdDevWholeNumbers, size=wholeNumbersLabelToChange)
    #percentageNumbersLabelNoise = np.random.normal(mean, stdDevPercentages, size=percentageNumbersLabelToChange)
    #
    #noisyLabel[wholeNumbersLabelMask     ] = np.maximum(0, noisyLabel[wholeNumbersLabelMask     ] + wholeNumbersLabelNoise)
    #noisyLabel[percentageNumbersLabelMask] = np.maximum(0, noisyLabel[percentageNumbersLabelMask] + percentageNumbersLabelNoise)
    #
    #for i in range(len(noisyLabel)):
    #    if wholeNumbersLabelMask[i] == True:
    #        noisyLabel[i] = math.ceil(noisyLabel[i])
            
    noisyPointWindowed = noisyPoint.reshape(3, 34)
    
    return noisyPointWindowed, noisyLabel

In [ ]:
numClusters = 4

interpolatedFeatures = []
interpolatedLabels   = []

for i in range(len(binIndexes)):
    binCondition = (fullLabelsArrayBinned[:, 0] == binIndexes[i][0]) & (fullLabelsArrayBinned[:, 1] == binIndexes[i][1]) & (fullLabelsArrayBinned[:, 2] == binIndexes[i][2]) & (fullLabelsArrayBinned[:, 3] == binIndexes[i][3]) & (fullLabelsArrayBinned[:, 4] == binIndexes[i][4])
    
    matchingLabelIndices = np.where(binCondition)[0]
    
    matchingFeatures = fullFeaturesArray[matchingLabelIndices]
    matchingLabels   = fullLabelsArray  [matchingLabelIndices]

    #if matching features is less than number of clusters, we either need to reduce the size of the clusters
    #or if its size 1, add gaussian noise n times
    
    matchingFeatureLen   = matchingFeatures.shape[0]
    matchingFeatures2d   = matchingFeatures.reshape(matchingFeatureLen, -1)
    
    actualNumClusters = numClusters
    
    if len(matchingFeatures2d) < 4:
        actualNumClusters = 1
        
    neededPoints = 703 - len(matchingFeatures2d)
    
    if neededPoints == 0:
        continue
    
    #if a cluster has length 1, we should exclude it and only use those which we can interpolate new points from
    kmeans = KMeans(n_clusters=actualNumClusters, random_state=0)
    
    kmeans.fit(matchingFeatures2d)
    
    validClusterIndices = []
    
    for j in range(numClusters):
        clusterIndices = np.where(kmeans.labels_ == j)[0]
        numFeaturesInCluster = len(clusterIndices)
        
        #print(numFeaturesInCluster)
        
        if numFeaturesInCluster > 1:
            validClusterIndices.append(j)
            
    numValidClusters = len(validClusterIndices)
    
    if numValidClusters == 0:
        #print(f"smote naive: {len(matchingFeatures2d)}, {neededPoints}")
        
        while neededPoints > 0:
            noisyPoint, noisyLabel = addNoiseToPoints(matchingFeatures2d[0], matchingLabels[0])
            
            interpolatedFeatures.append(noisyPoint.tolist())
            interpolatedLabels  .append(noisyLabel.tolist())
            
            neededPoints -=1
            
        #print(len(interpolatedFeatures))
        #print(len(interpolatedLabels))
             
    else:
        neededPointsPerCluster = math.floor(neededPoints / numValidClusters)
        
        #print(f"smote unaive: {len(matchingFeatures2d)}, {neededPoints}, {neededPointsPerCluster}")
        
        newPointsInBin = 0
        
        for j in range(numValidClusters):
            clusterIndices            =  np.where(kmeans.labels_ == validClusterIndices[j])[0]
            matchingFeatures2dCluster = matchingFeatures2d[clusterIndices]
            matchingLabelsCluster     = matchingLabels    [clusterIndices]
            
            clusterLength = len(matchingFeatures2dCluster)
            pairsInCluster = (clusterLength * (clusterLength - 1))/2
            
            #print(f"cluster length: {clusterLength}")
            
            clusterRun = math.ceil(neededPointsPerCluster / pairsInCluster)
            
            for k in range(len(matchingFeatures2dCluster)):
                for l in range(k+1, len(matchingFeatures2dCluster)):
                    firstFeature = matchingFeatures2dCluster[k]
                    firstLabel   = matchingLabelsCluster    [k]
                    
                    secondFeature = matchingFeatures2dCluster[l]
                    secondLabel   = matchingLabelsCluster    [l]
                    
                    pointsNeeded = clusterRun
                    
                    stepSize = 1.0 / (pointsNeeded + 1.0)
                    
                    interpolatedFeature = []
                    
                    #also need to check if we've already gotten enough points for this cluster
                    #if either point has masked values, prefer gaussian noise of the one without over interpolating
                    #if both points have masked values, fill in all zeros
                    while pointsNeeded > 0 and newPointsInBin < neededPoints:
                        interpolatedFeatureWindow = []
                        
                        for m in range(len(firstFeature)):
                            if m == 34 or m == 68:
                                interpolatedFeature.append(interpolatedFeatureWindow)
                                
                                interpolatedFeatureWindow = []
                            
                            firstFeaturePoint  = firstFeature [m]
                            secondFeaturePoint = secondFeature[m]

                            if firstFeaturePoint == -10000.0 and secondFeaturePoint != -10000.0:
                                noisySecondFeature,noisySecondLabel = addNoiseToPoints(secondFeature, secondLabel)

                                if m == 0:
                                    interpolatedFeature.append(noisySecondFeature[0])
                                    interpolatedFeature.append(noisySecondFeature[1])
                                    interpolatedFeature.append(noisySecondFeature[2])
                                elif m == 34:
                                    interpolatedFeature.append(noisySecondFeature[1])
                                    interpolatedFeature.append(noisySecondFeature[2])
                                else:
                                    interpolatedFeature.append(noisySecondFeature[2])
                                    
                                break

                                #print(interpolatedFeature)

                                #raise ExceptionType("An optional error message")

                            elif secondFeaturePoint == -10000.0 and firstFeaturePoint != -10000.0:
                                noisyFirstFeature,noisyFirstLabel = addNoiseToPoints(firstFeature, firstLabel)

                                if m == 0:
                                    interpolatedFeature.append(noisyFirstFeature[0])
                                    interpolatedFeature.append(noisyFirstFeature[1])
                                    interpolatedFeature.append(noisyFirstFeature[2])
                                elif m == 34:
                                    interpolatedFeature.append(noisyFirstFeature[1])
                                    interpolatedFeature.append(noisyFirstFeature[2])
                                else:
                                    interpolatedFeature.append(noisyFirstFeature[2])
                                    
                                break
                                
                                #print(noisyFirstFeature[0])
                                # print(f"{m}, {firstFeaturePoint}, {secondFeaturePoint}")
                                #print(interpolatedFeature)

                                #raise ExceptionType("An optional error message")
                            
                            else:
                                slope = max(secondFeaturePoint, firstFeaturePoint) - min(secondFeaturePoint, firstFeaturePoint)
                                
                                interpolatedPoint = min(secondFeaturePoint, firstFeaturePoint) + slope * pointsNeeded * stepSize
                                
                                if m <= 19 or (m >= 34 and m < 54) or (m >= 68 and m < 88):
                                    interpolatedPoint = math.floor(interpolatedPoint)
                                    
                                interpolatedFeatureWindow.append(interpolatedPoint)
                                
                                #print(f"{i}, {firstFeaturePoint}, {secondFeaturePoint}, {slope}, {pointsNeeded}, {stepSize}, {interpolatedPoint}")

                        if len(interpolatedFeature) < 3:
                            interpolatedFeature.append(interpolatedFeatureWindow)

                        startAge = interpolatedFeature[0][0]

                        if interpolatedFeature[1][0] != -10000.0:
                            interpolatedFeature[1][0] = startAge + 1
                        if interpolatedFeature[2][0] != -10000.0:
                            interpolatedFeature[2][0] = startAge + 2
    
                        interpolatedLabel = []
                        
                        for m in range(len(firstLabel)):
                            firstLabelPoint  = firstLabel [m]
                            secondLabelPoint = secondLabel[m]
                            
                            slope = max(secondLabelPoint, firstLabelPoint) - min(secondLabelPoint, firstLabelPoint)
                            
                            interpolatedPoint = min(secondLabelPoint, firstLabelPoint) + slope * pointsNeeded * stepSize
                    
                            if m != 4:
                                interpolatedPoint = math.floor(interpolatedPoint)
                                
                            interpolatedLabel.append(interpolatedPoint)
                            
                            #print(f"{i}, {firstLabelPoint}, {secondLabelPoint}, {slope}, {interpolatedPoint}")
                            
                        pointsNeeded -= 1
                        newPointsInBin += 1

                        interpolatedFeatures.append(interpolatedFeature)
                        interpolatedLabels  .append(interpolatedLabel  )
                        
                        interpolatedFeature = []
                        
        #print(len(interpolatedFeatures))
        #print(len(interpolatedLabels))

In [ ]:
interpolatedFeatures[30000]

In [ ]:
#len(interpolatedFeatures[30000][2])
interpolatedFeatures[30000]

In [ ]:
interpolatedFeaturesArray = np.array(interpolatedFeatures)
interpolatedLabelsArray   = np.array(interpolatedLabels)

In [ ]:
concatenatedFeaturesArray = np.concatenate((fullFeaturesArray, interpolatedFeaturesArray), axis=0)
concatenatedLabelsArray   = np.concatenate((fullLabelsArray  , interpolatedLabelsArray  ), axis=0)

In [ ]:
concatenatedRunBins              = np.linspace(min(concatenatedLabelsArray[:,0]), max(concatenatedLabelsArray[:,0]), num=4)
concatenatedHomeRunsBins         = np.linspace(min(concatenatedLabelsArray[:,1]), max(concatenatedLabelsArray[:,1]), num=4)
concatenatedRbiBins              = np.linspace(min(concatenatedLabelsArray[:,2]), max(concatenatedLabelsArray[:,2]), num=4)
concatenatedStolenBaseBins       = np.linspace(min(concatenatedLabelsArray[:,3]), max(concatenatedLabelsArray[:,3]), num=4)
concatenatedOnBasePercentageBins = np.linspace(min(concatenatedLabelsArray[:,4]), max(concatenatedLabelsArray[:,4]), num=4)

concatenatedRunsBinned             = np.digitize(concatenatedLabelsArray[:,0], concatenatedRunBins, right=True)
concatenatedHomeRunsBinned         = np.digitize(concatenatedLabelsArray[:,1], concatenatedHomeRunsBins, right=True)
concatenatedRbisBinned             = np.digitize(concatenatedLabelsArray[:,2], concatenatedRbiBins, right=True)
concatenatedStolenBasesBinned      = np.digitize(concatenatedLabelsArray[:,3], concatenatedStolenBaseBins, right=True)
concatenatedOnBasePercentageBinned = np.digitize(concatenatedLabelsArray[:,4], concatenatedOnBasePercentageBins, right=True)

fullConcatenatedLabelsArrayBinned = np.column_stack((concatenatedRunsBinned, concatenatedHomeRunsBinned, concatenatedRbisBinned, concatenatedStolenBasesBinned, concatenatedOnBasePercentageBinned))

In [ ]:
concatedBinCount = {}

for label in fullConcatenatedLabelsArrayBinned:
    labelString = str(label[0]) + str(label[1]) + str(label[2]) + str(label[3]) + str(label[4])

    if labelString in concatedBinCount:
        concatedBinCount[labelString] = concatedBinCount[labelString] + 1
    else:
        concatedBinCount[labelString] = 1

In [ ]:
concatenatedLabels = list(concatedBinCount.keys())
concatenatedCount  = list(concatedBinCount.values())

plt.figure(figsize=(16, 12))
bar = plt.bar(concatenatedLabels, concatenatedCount)

_ = plt.xticks(concatenatedLabels, rotation="vertical")
_ = plt.bar_label(bar, concatenatedCount)

In [ ]:
#binnedDf = pd.DataFrame({
#    'runs'            : runsBinned,
#    'homeRuns'        : homeRunsBinned,
#    'rbis'            : rbisBinned,
#    'stolenBases'     : stolenBasesBinned,
#    'onBasePercentage': onBasePercentageBinned
#})
#
#uniqueRowsIndexes = binnedDf.drop_duplicates(keep=False).index
#
#uniqueRowsIndexesArray        = np.array(uniqueRowsIndexes)
#fullLabelsArrayBinnedFiltered = np.delete(fullLabelsArrayBinned, uniqueRowsIndexesArray, axis=0)
#fullFeaturesArrayFiltered     = np.delete(fullFeaturesArray, uniqueRowsIndexesArray, axis=0)
#fullLabelsArrayFiltered       = np.delete(fullLabelsArray, uniqueRowsIndexesArray, axis=0)

In [ ]:
concatenatedFeaturesArray[10]

In [ ]:
#NEED TO CHANGE IMPLEMEENTATION OR ORDERING OF MASKING TO AVOID NORMALIZING MASKED VALUES

featureMask = concatenatedFeaturesArray == -10000.0

maskedFeatureArray = np.ma.masked_array(concatenatedFeaturesArray, mask=featureMask)

featuresMean = maskedFeatureArray.mean(axis=(0,1), keepdims=True)
featuresStd  = maskedFeatureArray.std(axis=(0,1), keepdims=True)

labelMeansList = [[featuresMean[0][0][3], featuresMean[0][0][8], featuresMean[0][0][9], featuresMean[0][0][16], featuresMean[0][0][21]]]
labelStdList   = [[featuresStd [0][0][3], featuresStd [0][0][8], featuresStd [0][0][9], featuresStd [0][0][16], featuresStd [0][0][21]]]  

labelsMean = np.array(labelMeansList)
labelsStd  = np.array(labelStdList)

normalizedFeaturesMasked = (maskedFeatureArray - featuresMean) / (featuresStd)
normalizedLabels         = (concatenatedLabelsArray - labelsMean) / (labelsStd)

normalizedFeatures = normalizedFeaturesMasked.data

In [ ]:
normalizedFeatures[10]

In [ ]:
trainFeatures,valTestFeatures,trainLabels,valTestLables = train_test_split(normalizedFeatures, normalizedLabels, stratify=fullConcatenatedLabelsArrayBinned, test_size=0.2)

valFeatures,testFeatures,valLabels,testLabels = train_test_split(valTestFeatures, valTestLables, test_size=0.5)

In [ ]:
trainFeatures.shape

In [ ]:
#l2/l1 regularization, dropout, activation function, optmizier, loss function, hidden units, hidden layers, batch size, batch normalization
def getModelForTuner(hp):
    hp_learning_rate     = hp.Choice('learning_rate'    , values=[1e-2, 1e-3, 1e-4])
    hp_weight_decay      = hp.Choice('weight_decay'     , values=[1e-3,1e-4,1e-5])   

    num_rnn_layers = hp.Int("num_rnn_layers", min_value=1, max_value=3)
    
    model = tf.keras.Sequential()
    model.add(layers.Masking(mask_value=-10000.0, input_shape=(3, 32)))

    for i in range(num_rnn_layers):
        hp_units             = hp.Int   (f'units_{i}', min_value=5, max_value=30, step=6)
        hp_dropout           = hp.Choice(f'dropout_{i}'          , values=[0.2,0.3,0.4,0.5])     
        hp_recurrent_dropout = hp.Choice(f'recurrent_dropout_{i}', values=[0.2,0.3,0.4,0.5])  
        hp_regularizer       = hp.Choice(f'regularizer_{i}'      , values=[0.0001, 0.001, 0.01])  

        if i < num_rnn_layers - 1:
            model.add(layers.LSTM(units=hp_units, dropout=hp_dropout, recurrent_dropout=hp_recurrent_dropout, kernel_regularizer=regularizers.l2(hp_regularizer), return_sequences=True))
        else:
            model.add(layers.LSTM(units=hp_units, dropout=hp_dropout, recurrent_dropout=hp_recurrent_dropout, kernel_regularizer=regularizers.l2(hp_regularizer), return_sequences=False))
        
        model.add(layers.LayerNormalization())
        model.add(layers.Dense(units = 5))

    model.compile(optimizer = tf.keras.optimizers.Adam(learning_rate=hp_learning_rate, weight_decay=hp_weight_decay), loss=tf.keras.losses.Huber(), metrics=[tf.keras.metrics.R2Score(), "accuracy"])

    return model

In [ ]:
tuner = kt.Hyperband(getModelForTuner,
                     objective='val_loss',
                     max_epochs=100,
                     factor=3,
                     directory='tuner',
                     project_name='tuner_test')

In [ ]:
tuner.search(trainFeatures, trainLabels, batch_size=64, epochs=100, validation_data=(valFeatures, valLabels))

In [ ]:
best_hps=tuner.get_best_hyperparameters(num_trials=1)[0]

In [ ]:
print(best_hps.get('num_rnn_layers'))
print(best_hps.get('learning_rate'))
print(best_hps.get('weight_decay'))
print(best_hps.get('units_0'))
print(best_hps.get('dropout_0'))
print(best_hps.get('recurrent_dropout_0'))
print(best_hps.get('regularizer_0'))

print("-----------------")

print(best_hps.get('units_1'))
print(best_hps.get('dropout_1'))
print(best_hps.get('recurrent_dropout_1'))
print(best_hps.get('regularizer_1'))

print("--------------------------")

print(best_hps.get('units_2'))
print(best_hps.get('dropout_2'))
print(best_hps.get('recurrent_dropout_2'))
print(best_hps.get('regularizer_2'))


In [ ]:
trainFeatures.shape

In [ ]:
#l2/l1 regularization, dropout, activation function, optmizier, loss function, hidden units, hidden layers, batch size, batch normalization
def get_model():
    normalizeInput = layers.Normalization()
    normalizeInput.adapt(trainFeatures)

    model = tf.keras.Sequential([
        layers.Masking(mask_value=-10000.0, input_shape=(3, 34)),
        #normalizeInput,
        layers.LSTM(units=23, dropout=0.2, recurrent_dropout=0.2, kernel_regularizer=regularizers.l2(0.0001), return_sequences=True),
        layers.LayerNormalization(),
        layers.LSTM(units=29, dropout=0.3, recurrent_dropout=0.4, kernel_regularizer=regularizers.l2(0.0001), return_sequences=True),
        layers.LayerNormalization(),
        layers.LSTM(units=23, dropout=0.5, recurrent_dropout=0.5, kernel_regularizer=regularizers.l2(0.0001), return_sequences=False),
        layers.LayerNormalization(),
        layers.Dense(units = 5)
    ])

    model.compile(optimizer = tf.keras.optimizers.Adam(learning_rate=0.001, weight_decay=1e-05), loss=tf.keras.losses.Huber(), metrics=[tf.keras.metrics.R2Score(), "accuracy"])

    return model

In [ ]:
def get_bidirectional_model():
    model = tf.keras.Sequential([
        layers.Masking(mask_value=-10000.0, input_shape=(3, 34)),
        #normalizeInput,
        layers.Bidirectional(layers.LSTM(units=23, dropout=0.2, recurrent_dropout=0.2, kernel_regularizer=regularizers.l2(0.0001), return_sequences=True)),
        layers.LayerNormalization(),
        layers.Bidirectional(layers.LSTM(units=29, dropout=0.3, recurrent_dropout=0.4, kernel_regularizer=regularizers.l2(0.0001), return_sequences=True)),
        layers.LayerNormalization(),
        layers.Bidirectional(layers.LSTM(units=23, dropout=0.5, recurrent_dropout=0.5, kernel_regularizer=regularizers.l2(0.0001), return_sequences=False)),
        layers.LayerNormalization(),
        layers.Dense(units = 5)
    ])

    model.compile(optimizer = tf.keras.optimizers.Adam(learning_rate=0.001, weight_decay=1e-05), loss=tf.keras.losses.Huber(), metrics=[tf.keras.metrics.R2Score(), "accuracy"])

    return model

In [ ]:
curModel = get_bidirectional_model()

curModel.summary()

In [ ]:
curModel.fit(trainFeatures, trainLabels, batch_size=64, epochs=100, validation_data=(valFeatures, valLabels))

In [ ]:
accuracy: 0.7168 - loss: 0.1594 - r2_score: 0.7955 - val_accuracy: 0.7590 - val_loss: 0.1356 - val_r2_score: 0.8308

In [ ]:
7ms/step - accuracy: 0.7340 - loss: 0.1404 - r2_score: 0.8011 - val_accuracy: 0.7956 - val_loss: 0.1095 - val_r2_score: 0.8472

In [ ]:
baselineResults = curModel.evaluate(testFeatures, testLabels)

In [ ]:
accuracy: 0.7432 - loss: 0.1335 - r2_score: 0.8316

accuracy: 0.7898 - loss: 0.1080 - r2_score: 0.8462

In [ ]:
accuracy: 0.8001 - loss: 0.1108 - r2_score: 0.8425

In [ ]:
curModel.save_weights('./modelCheckPoints/model_1-30-2025_2.weights.h5')

In [ ]:
for featureIndex in range(testFeatures.shape[2]):
   print(df.columns[featureIndex+2])
   shuffledTestFeatures = testFeatures.copy()

   testColumnElements = [shuffledTestFeatures[i][j][featureIndex] for i in range(len(shuffledTestFeatures)) for j in range(len(shuffledTestFeatures[i]))]

   random.shuffle(testColumnElements)
   
   k = 0
   
   for i in range(len(shuffledTestFeatures)):
       for j in range(len(shuffledTestFeatures[i])):
           shuffledTestFeatures[i][j][featureIndex] = testColumnElements[k]
   
           k += 1

   shuffledResults = curModel.evaluate(shuffledTestFeatures, testLabels)

   print(shuffledResults[1] / baselineResults[1])

In [ ]:
#TestResults
playerFeaturesQuery = "SELECT Bios.playerId,season,Bios.dob,plateAppearances,atBats,runs,hits,singles,doubles,triples,homeRuns,rbis,sacHits,sacFlies,hitByPitch,walks,intentionalWalks,strikeOuts,stolenBases,caughtStealing,groundedIntoDoublePlays,catcherInterference,battingAverage,onBasePercentage,sluggingPercentage,onBasePlusSlugging,runsPerTob,rbisPerBip,babip,spd,groundBallPercentage,flyBallPercentage,lineDrivePercentage,popUpPercentage,hardHitPercentage,barrelPercentage FROM SeasonStatsHitting INNER JOIN Bios on Bios.playerId = SeasonStatsHitting.playerId WHERE season BETWEEN 2023 and 2025 and SeasonStatsHitting.playerId =\"682829\" ORDER BY SeasonStatsHitting.playerId,season"

playerDf = pd.read_sql(playerFeaturesQuery, engine)

In [ ]:
playerDf['dob'] = playerDf.apply(lambda row: calculate_age_from_timestamps(row['dob'], seasonToStartTimeStamp[row['season']]), axis=1)

In [ ]:
playerDf.transpose()

In [ ]:
playerStats = []

playerFrameArray = playerDf.values.tolist()

for i in range(len(playerFrameArray)):
    playerStats.append(playerFrameArray[i][2:])

if len(playerStats) < 3:
    diff = 3 - len(playerStats)

    while diff > 0:
        playerStats.append([-10000.0] * 34)

        diff -= 1

In [ ]:
playerWindow = np.array([playerStats])

playerWindowMask = playerWindow == -10000.0

maskedPlayerWindow = np.ma.masked_array(playerWindow, mask=playerWindowMask)

playerWindowMaskNormalized = (maskedPlayerWindow - featuresMean) / (featuresStd)

playerWindowNormalized = playerWindowMaskNormalized.data

In [ ]:
playerWindowNormalized

In [ ]:
playerPrediction = curModel.predict(playerWindowNormalized)

In [ ]:
playerPrediction

In [ ]:
indexToMean = {}

indexToMean[0] = labelsMean[0][0]
indexToMean[1] = labelsMean[0][1]
indexToMean[2] = labelsMean[0][2]
indexToMean[3] = labelsMean[0][3]
indexToMean[4] = labelsMean[0][4]

indexToStd = {}

indexToStd[0] = labelsStd[0][0]
indexToStd[1] = labelsStd[0][1]
indexToStd[2] = labelsStd[0][2]
indexToStd[3] = labelsStd[0][3]
indexToStd[4] = labelsStd[0][4]

for i in range(len(playerPrediction[0])):
    prediction = playerPrediction[0][i]

    curMean = indexToMean[i]
    curStd  = indexToStd [i]

    #print(f"{curMean}, {curStd}")

    prediction = prediction * curStd + curMean

    #value * std of column + mean of column
    #prediction = math.exp(prediction) - 1

    if i < 4:
        print(math.ceil(prediction))
    else:
        print(round(prediction, 3))

In [ ]:
curBestModel = get_bidirectional_model()

curBestModel.load_weights('./modelCheckPoints/model_1-30-2025_2.weights.h5')

In [ ]:
#763

season = 2025

hitterIds = []

hitterIdQuery = f"SELECT DISTINCT(PerGameStatsHitting.playerId),firstName,lastName from Games INNER JOIN PerGameStatsHitting ON Games.gameId = PerGameStatsHitting.gameId INNER JOIN Bios ON Bios.playerId = PerGameStatsHitting.playerId WHERE Games.season={season}"

connection = sqlite3.connect("C:/baseball_info.db")
cursor     = connection.cursor()

cursor.execute(hitterIdQuery)

rows = cursor.fetchall()

for row in rows:
    hitterIds.append((row[0],row[1],row[2]))

cursor    .close()
connection.close()

print(len(hitterIds))

In [ ]:
hitterIds[0]

In [ ]:
indexToMean = {}

indexToMean[0] = labelsMean[0][0]
indexToMean[1] = labelsMean[0][1]
indexToMean[2] = labelsMean[0][2]
indexToMean[3] = labelsMean[0][3]
indexToMean[4] = labelsMean[0][4]

indexToStd = {}

indexToStd[0] = labelsStd[0][0]
indexToStd[1] = labelsStd[0][1]
indexToStd[2] = labelsStd[0][2]
indexToStd[3] = labelsStd[0][3]
indexToStd[4] = labelsStd[0][4]

In [ ]:
playerPredictions = []

for hitter in hitterIds:
    playerId = hitter[0]

    playerFeaturesQuery = f"SELECT Bios.playerId,season,Bios.dob,plateAppearances,atBats,runs,hits,singles,doubles,triples,homeRuns,rbis,sacHits,sacFlies,hitByPitch,walks,intentionalWalks,strikeOuts,stolenBases,caughtStealing,groundedIntoDoublePlays,catcherInterference,battingAverage,onBasePercentage,sluggingPercentage,onBasePlusSlugging,runsPerTob,rbisPerBip,babip,spd,groundBallPercentage,flyBallPercentage,lineDrivePercentage,popUpPercentage,hardHitPercentage,barrelPercentage FROM SeasonStatsHitting INNER JOIN Bios on Bios.playerId = SeasonStatsHitting.playerId WHERE season BETWEEN 2023 and 2025 and SeasonStatsHitting.playerId =\"{playerId}\" ORDER BY SeasonStatsHitting.playerId,season"

    playerDf = pd.read_sql(playerFeaturesQuery, engine)

    playerDf['dob'] = playerDf.apply(lambda row: calculate_age_from_timestamps(row['dob'], seasonToStartTimeStamp[row['season']]), axis=1)

    playerStats = []

    playerFrameArray = playerDf.values.tolist()
    
    for i in range(len(playerFrameArray)):
        playerStats.append(playerFrameArray[i][2:])
    
    if len(playerStats) < 3:
        diff = 3 - len(playerStats)
    
        while diff > 0:
            playerStats.append([-10000.0] * 34)
    
            diff -= 1

    playerWindow = np.array([playerStats])

    playerWindowMask = playerWindow == -10000.0
    
    maskedPlayerWindow = np.ma.masked_array(playerWindow, mask=playerWindowMask)
    
    playerWindowMaskNormalized = (maskedPlayerWindow - featuresMean) / (featuresStd)
    
    playerWindowNormalized = playerWindowMaskNormalized.data

    playerPrediction = curBestModel.predict(playerWindowNormalized)

    normalizedPlayerPrediction = (playerId, hitter[1], hitter[2])

    for i in range(len(playerPrediction[0])):
        prediction = playerPrediction[0][i]
    
        curMean = indexToMean[i]
        curStd  = indexToStd [i]
    
        prediction = prediction * curStd + curMean
    
        if i < 4:
            prediction = max(0, math.ceil(prediction))
        else:
            prediction = max(0.0, round(prediction, 3))

        normalizedPlayerPrediction += (prediction,)

    playerPredictions.append(normalizedPlayerPrediction)

In [ ]:
print(len(playerPredictions))

In [ ]:
playerPredictions

In [ ]:
rbiSortedPredictions = sorted(playerPredictions, key=lambda player: player[5], reverse=True)

In [ ]:
for i in range(20):
    print(rbiSortedPredictions[i])

In [ ]:
maxRuns = 0.0
minRuns = 1000.0

maxHomeRuns = 0.0
minHomeRuns = 1000.0

maxRbis = 0.0
minRbis = 1000.0

maxStolenBases = 0.0
minStolenBases = 1000.0

maxOnBasePercentage = 0.0
minOnBasePercentage = 1.1

for player in playerPredictions:
    runs             = player[3]
    homeRuns         = player[4]
    rbis             = player[5]
    stolenBases      = player[6]
    onBasePercentage = player[7]

    if runs == -1:
        print(player)

    maxRuns = max(maxRuns, runs)
    minRuns = min(minRuns, runs)

    maxHomeRuns = max(maxHomeRuns, homeRuns)
    minHomeRuns = min(minHomeRuns, homeRuns)

    maxRbis = max(maxRbis, rbis)
    minRbis = min(minRbis, rbis)

    maxStolenBases = max(maxStolenBases, stolenBases)
    minStolenBases = min(minStolenBases, stolenBases)

    maxOnBasePercentage = max(maxOnBasePercentage, onBasePercentage)
    minOnBasePercentage = min(minOnBasePercentage, onBasePercentage)

print(f"{maxRuns}, {minRuns}")
print(f"{maxHomeRuns}, {minHomeRuns}")
print(f"{maxRbis}, {minRbis}")
print(f"{maxStolenBases}, {minStolenBases}")
print(f"{maxOnBasePercentage}, {minOnBasePercentage}")

In [ ]:
def normalize(num, maxValue, minValue):
    if maxValue == 0 and minValue == 0:
        return 0
    if maxValue == minValue:
        return 0
    
    normalizedValue = (num - minValue) / (maxValue - minValue)

    return normalizedValue

In [ ]:
for i in range(len(playerPredictions)):
    player = playerPredictions[i]
    
    runs             = player[3]
    homeRuns         = player[4]
    rbis             = player[5]
    stolenBases      = player[6]
    onBasePercentage = player[7]

    normalizedRuns        = normalize(runs, maxRuns, minRuns)
    normalizedHrs         = normalize(homeRuns, maxHomeRuns, minHomeRuns)
    normalizedObp         = normalize(onBasePercentage, maxOnBasePercentage, minOnBasePercentage)
    normalizedStolenBases = normalize(stolenBases, maxStolenBases, minStolenBases)
    normalizedRbis        = normalize(rbis, maxRbis, minRbis)

    overallGrade = normalizedRuns + normalizedHrs + normalizedObp + normalizedStolenBases + normalizedRbis

    player += (overallGrade,)

    playerPredictions[i] = player

In [ ]:
gradeSortedPredictions = sorted(playerPredictions, key=lambda player: player[8], reverse=True)

In [ ]:
for i in range(50):
    print(gradeSortedPredictions[i])

In [ ]:
predictionHeaders = ["id", "firstName", "lastName", "runs", "homeRuns", "rbis", "stolenBases", "onBasePercentage", "grade"]

filePath = "./batter_predictions_2026.csv"

predictionsDf = pd.DataFrame(gradeSortedPredictions, columns=predictionHeaders)

predictionsDf.to_csv(filePath, index=False)

In [ ]:
## maxDifferenceRuns = 0.0
minDifferenceRuns = 100.0

maxDifferenceHomeRuns = 0.0
minDifferenceHomeRuns = 100.0

maxDifferenceRbis = 0.0
minDifferenceRbis = 100.0

maxDifferenceStolenBases = 0.0
minDifferenceStolenBases = 100.0

sumDifferencesRuns        = 0.0
sumDifferencesHomeRuns    = 0.0
sumDifferencesRbis        = 0.0
sumDifferencesStolenBases = 0.0

maxChangeRunsIndex = -1

for i in range(len(concatenatedFeaturesArray)):
    feature = concatenatedFeaturesArray[i]
    label   = concatenatedLabelsArray  [i]

    lastFeature = None

    if feature[2][0] != -10000.0:
        lastFeature = feature[2]
    elif feature[1][0] != -10000.0:
        lastFeature = feature[1]
    else:
        lastFeature = feature[0]

    #[[featuresMean[0][0][3], featuresMean[0][0][8], featuresMean[0][0][9], featuresMean[0][0][16], featuresMean[0][0][21]]]
    
    featureRuns     = lastFeature[3]
    featureHomeRuns = lastFeature[8]
    featureRbis     = lastFeature[9]
    featureSbs      = lastFeature[16]
    featureObp      = lastFeature[21]

    differenceRuns        = abs(label[0] - featureRuns    )
    differenceHomeRuns    = abs(label[1] - featureHomeRuns)
    differenceRbis        = abs(label[2] - featureRbis    )
    differenceStolenBases = abs(label[3] - featureSbs     )

    averageRuns        = (featureRuns     + label[0]) / 2.0
    averageHomeRuns    = (featureHomeRuns + label[1]) / 2.0
    averageRbis        = (featureRbis     + label[2]) / 2.0
    averageStolenBases = (featureSbs      + label[3]) / 2.0

    percentageDifferenceRuns        = (differenceRuns        / averageRuns       ) * 100.0 if averageRuns        != 0.0 else 0.0
    percentageDifferenceHomeRuns    = (differenceHomeRuns    / averageHomeRuns   ) * 100.0 if averageHomeRuns    != 0.0 else 0.0
    percentageDifferencebis         = (differenceRbis        / averageRbis       ) * 100.0 if averageRbis        != 0.0 else 0.0
    percentageDifferenceStolenBases = (differenceStolenBases / averageStolenBases) * 100.0 if averageStolenBases != 0.0 else 0.0

    if percentageDifferenceRuns > maxDifferenceRuns:
        maxChangeRunsIndex = i
        print(f"{featureRuns}, {label[0]}")
        
    maxDifferenceRuns = max(maxDifferenceRuns, percentageDifferenceRuns)
    minDifferenceRuns = min(minDifferenceRuns, percentageDifferenceRuns)

    maxDifferenceHomeRuns = max(maxDifferenceHomeRuns, percentageDifferenceHomeRuns)
    minDifferenceHomeRuns = min(minDifferenceHomeRuns, percentageDifferenceHomeRuns)

    maxDifferenceRbis = max(maxDifferenceRbis, percentageDifferencebis)
    minDifferenceRbis = min(minDifferenceRbis, percentageDifferencebis)

    maxDifferenceStolenBases = max(maxDifferenceStolenBases, percentageDifferenceStolenBases)
    minDifferenceStolenBases = min(minDifferenceStolenBases, percentageDifferenceStolenBases)

    sumDifferencesRuns        += percentageDifferenceRuns       
    sumDifferencesHomeRuns    += percentageDifferenceHomeRuns   
    sumDifferencesRbis        += percentageDifferencebis        
    sumDifferencesStolenBases += percentageDifferenceStolenBases

In [ ]:
averageDifferenceRuns        = sumDifferencesRuns        / len(concatenatedFeaturesArray)
averageDifferenceHomeRuns    = sumDifferencesHomeRuns    / len(concatenatedFeaturesArray)
averageDifferenceRbis        = sumDifferencesRbis        / len(concatenatedFeaturesArray)
averageDifferenceStolenBases = sumDifferencesStolenBases / len(concatenatedFeaturesArray)

print(f"RUNS: {maxChangeRunsIndex}, {maxDifferenceRuns}, {minDifferenceRuns}, {averageDifferenceRuns}")
print(f"HOME RUNS: {maxDifferenceHomeRuns}, {minDifferenceHomeRuns}, {averageDifferenceHomeRuns}")
print(f"RBIS: {maxDifferenceRbis}, {minDifferenceRbis}, {averageDifferenceRbis}")
print(f"STOLEN BASES: {maxDifferenceStolenBases}, {minDifferenceStolenBases}, {averageDifferenceStolenBases}")

In [ ]:
bucket features by percentages gained or lost in all valid categories, then use smote to even out the buckets

after this use smote to bucket labelss and create new data (ie what we're already doing)

In [ ]:
https://en.wikipedia.org/wiki/Speed_Score#:~:text=Factor%201%20(Stolen%20base%20percentage,:%203B:%20SS:%20OF: